# SkillSync — NLP-Powered Resume & Job Description Matching Platform
## Comprehensive Analysis, Architecture Demonstration & Model Verification

**Author / Academic Submission:** SkillSync Engineering Team  
**Course / Context:** Natural Language Processing & Machine Learning Project  
**Version:** 2.1.0  

---

### 1. Project Introduction

**Problem Statement:**  
Recruitment systems and candidates often struggle with asymmetric information: candidates don't know why their resumes are filtered out by Applicant Tracking Systems (ATS), while recruiters spend hundreds of hours manually screening resumes against job descriptions (JDs). Many existing matching tools rely on simple keyword counting, yielding false positives (e.g., mistaking `C` for `C++` or treating a movie title as a skill) or inflated 100% match scores that mask severe technical skill gaps.

**Project Objectives:**  
1. **Accurate Skill Extraction:** Extract technical skills and competencies from resumes and JDs using boundary-aware regex patterns and canonicalization.
2. **Domain-Isolated Scoring:** Reward explicit required skill matches ($R$), domain-relevant peripheral competencies ($A$), and deep textual context ($T$).
3. **Explainability & Transparency:** Avoid black-box scoring by providing full breakdowns of matched, missing, and additional relevant skills.
4. **High Performance:** Optimize NLP preprocessing through cached WordNet lemmatization and precompiled regexes without heavy GPU/LLM dependencies.

**Technologies Used:**  
- **Python 3.11 / 3.12**
- **NLTK (Natural Language Toolkit):** Tokenization, stop words removal, cached WordNet lemmatizer.
- **Scikit-Learn:** TF-IDF Vectorization (`ngram_range=(1, 2)`, `sublinear_tf=True`), Cosine Similarity.
- **PyPDF:** PDF document parsing and text stream extraction.
- **Pandas & NumPy:** Dataset loading, tabular analysis, and numerical normalization.
- **Streamlit:** Interactive web interface for recruitment analytics.


---
### 2. System Architecture

SkillSync implements a multi-stage deterministic NLP pipeline:

```text
Resume PDF + Job Description
        │
        ├──> [PDF Text Extraction] (PyPDF + OCR/Ligature repair)
        │
        ├──> [NLP Preprocessing] (Tokenization, stopword removal, cached lemmatization)
        │
        ├──> [Skill Extraction & Canonicalization] (Boundary regex + Alias resolution)
        │
        ├──> [JD Requirement Extraction] (Separates required vs descriptive text)
        │
        ├──> [Required Skill Matching (R)] -> Matched & Missing skills
        │
        ├──> [Role-Aware Additional Relevance (A)] -> Domain clusters & cap limit
        │
        ├──> [TF-IDF Vectorization & Cosine Similarity (T)] -> Sublinear unigrams/bigrams
        │
        └──> [Composite Weighted Scoring] -> Final Score = 0.70*R + 0.10*A + 0.20*T
                   │
                   └──> [Verdict Classification] (Strong, Moderate, Fair, Low Match)
```


---
### 3. Library Imports & Environment Setup

We import project dependencies and ensure project root modules (`nlp`, `taxonomy`) are accessible.


In [ ]:
import sys
from pathlib import Path

# Ensure project root is in sys.path
project_root = Path.cwd().resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import re
import pickle
import numpy as np
import pandas as pd
import sklearn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import nlp
from nlp import (
    preprocess_text,
    clean_text,
    normalize_for_skill_search,
    canonicalize_skill,
    extract_skills,
    extract_jd_requirements,
    calculate_skill_match,
    calculate_similarity,
    calculate_final_score,
    get_verdict,
    extract_pdf_text,
)
from taxonomy import SKILLS, SKILL_ALIASES, ROLE_TAXONOMY, get_relevant_skills_for_jd

print(f"Python Version: {sys.version.split()[0]}")
print(f"Scikit-Learn Version: {sklearn.__version__}")
print(f"Total Canonical Skills in Taxonomy: {len(SKILLS)}")
print(f"Total Aliases in Taxonomy: {len(SKILL_ALIASES)}")


---
### 4. Dataset & Demonstration Inputs

SkillSync was developed with reference to the Kaggle Resume Dataset (`data/resume-data/Resume/Resume.csv`), comprising 2,400+ categorized resumes across 24 professional domains.
For reproducible academic demonstration, we define an anonymized software engineer resume and target role job descriptions.


In [ ]:
# Representative anonymized technical resume
SAMPLE_RESUME = """
Alex Morgan — Senior Software Engineer
Contact: alex.morgan@example.com | Portfolio: alexmorgan.dev | GitHub: alexmorgan

PROFESSIONAL SUMMARY:
Full-stack software engineer with 4+ years of experience designing scalable web services,
cloud infrastructure, and distributed data systems. Proficient in Python, TypeScript, and modern APIs.

EXPERIENCE:
Lead Backend Engineer — TechCorp (2022 - Present)
- Designed and deployed microservices using FastAPI, Python, and PostgreSQL, serving 1.5M monthly requests.
- Containerized application workflows using Docker and orchestrated automated CI/CD deployment via GitHub Actions.
- Implemented Redis caching layers and secure authentication with OAuth 2.0, reducing API latency by 45%.
- Engineered Next.js frontend interfaces integrated with REST APIs and WebSocket real-time updates.

TECHNICAL SKILLS:
- Languages: Python, TypeScript, JavaScript, SQL
- Frameworks: FastAPI, Express, React, Next.js, Node.js
- Databases: PostgreSQL, MongoDB, Redis
- Cloud & DevOps: Docker, AWS (EC2, S3), CI/CD, Git, Linux
- Core Concepts: REST APIs, Microservices, OAuth 2.0, Database Design
"""

# Target Full-Stack Job Description
SAMPLE_JD = """
Senior Full Stack Developer
We are seeking a talented Senior Full Stack Developer to build enterprise web applications.
Requirements:
- Proficient in React, Next.js, TypeScript, and Node.js.
- Strong backend development experience with Express, REST APIs, and GraphQL.
- Hands-on database design using PostgreSQL, MongoDB, and Redis.
- Solid understanding of Docker, CI/CD pipelines, and cloud deployment.
- Experience with Tailwind CSS.
"""

print(f"Sample Resume Length: {len(SAMPLE_RESUME)} chars")
print(f"Sample JD Length: {len(SAMPLE_JD)} chars")


---
### 5. PDF Text Extraction

SkillSync extracts text streams using `pypdf.PdfReader` with built-in font ligature and OCR split repair heuristics (e.g. repairing `T ypeScript` to `TypeScript` and `NL TK` to `NLTK`).


In [ ]:
# Inspect extract_pdf_text implementation
import inspect
print(inspect.getsource(extract_pdf_text))


---
### 6. NLP Text Preprocessing

The preprocessing pipeline cleans, tokenizes, removes English stop words, and lemmatizes tokens to base forms.
To achieve millisecond-level execution in web environments, `WordNetLemmatizer.lemmatize` is accelerated using an LRU cache (`@lru_cache(maxsize=20000)`).


In [ ]:
raw_snippet = "Developed scalable microservices with Python, building resilient APIs and databases!"
tokens = preprocess_text(raw_snippet)
cleaned_str = clean_text(raw_snippet)

print("Raw Input:     ", raw_snippet)
print("Tokens:        ", tokens)
print("Cleaned String:", cleaned_str)


---
### 7. Skill Extraction Engine

SkillSync uses boundary-isolated regular expressions compiled from `taxonomy.py`. Ambiguous single-letter and multiword skills are validated through contextual filters (`_is_valid_skill_context`).


In [ ]:
extracted_resume_skills = extract_skills(SAMPLE_RESUME)
print(f"Extracted {len(extracted_resume_skills)} skills from sample resume:")
print(extracted_resume_skills)


---
### 8. Skill Normalization & Disambiguation

To prevent score inflation from synonyms or plural forms, SkillSync maps variants to canonical keys:
- `Tailwind` vs `Tailwind CSS` $\to$ both canonicalize to `tailwind css`.
- `REST API` vs `REST APIs` $\to$ both canonicalize to `rest apis`.
- `Postgres` vs `PostgreSQL` vs `MySQL` $\to$ normalized to `sql`.
- Boundary lookaheads prevent `C` from triggering on `C++` or `C#`.


In [ ]:
test_cases = [
    ('Tailwind and Tailwind CSS', 'tailwind css'),
    ('REST API endpoints and REST APIs', 'rest apis'),
    ('Proficient in C++ and C# but never worked in raw C', 'c++ vs c'),
]

for phrase, desc in test_cases:
    found = extract_skills(phrase)
    print(f"{desc:30} -> Input: '{phrase}' => Found: {found}")


---
### 9. Job Requirement Extraction

Job descriptions are parsed to extract explicit technical requirements while discarding generic recruiter boilerplate.


In [ ]:
jd_requirements = extract_jd_requirements(SAMPLE_JD)
print(f"Extracted {len(jd_requirements)} technical requirements from JD:")
print(jd_requirements)


---
### 10. TF-IDF Vectorization

SkillSync utilizes `TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)`:
- **N-grams (1, 2):** Captures unigrams (`python`) and bigrams (`machine learning`, `rest apis`).
- **Sublinear TF:** Replaces term frequency $tf$ with $1 + \log(tf)$ to diminish the impact of repetitive buzzwords.


In [ ]:
resume_clean = clean_text(SAMPLE_RESUME)
jd_clean = clean_text(SAMPLE_JD)

vectorizer = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)
tfidf_matrix = vectorizer.fit_transform([jd_clean, resume_clean])

print(f"TF-IDF Matrix Shape: {tfidf_matrix.shape} (2 documents x {tfidf_matrix.shape[1]} n-gram features)")
print(f"Sample Feature Names: {vectorizer.get_feature_names_out()[:10]}")


---
### 11. Cosine Similarity & TF-IDF Score ($T$)

Cosine similarity measures the angular distance between vectors:
$$\text{Cosine Similarity}(\mathbf{u}, \mathbf{v}) = \frac{\mathbf{u} \cdot \mathbf{v}}{\|\mathbf{u}\| \|\mathbf{v}\|}$$
SkillSync combines 70% JD term coverage with 30% traditional cosine similarity to evaluate textual relevance without penalizing concise resumes.


In [ ]:
sim_score = calculate_similarity(SAMPLE_RESUME, SAMPLE_JD)
print(f"Computed TF-IDF Similarity Score (T): {sim_score:.2f}%")


---
### 12. Required Skill Matching ($R$)

Calculates the exact overlap between required JD skills and detected resume skills:
$$R = \left(\frac{|\text{Resume Skills} \cap \text{JD Requirements}|}{|\text{JD Requirements}|}\right) \times 100$$


In [ ]:
skill_match_res = calculate_skill_match(SAMPLE_RESUME, SAMPLE_JD)
print(f"Required Skills Match Score (R): {skill_match_res['skill_score']:.2f}%")
print(f"Matched Skills ({len(skill_match_res['matched_skills'])}): {skill_match_res['matched_skills']}")
print(f"Missing Skills ({len(skill_match_res['missing_skills'])}): {skill_match_res['missing_skills']}")


---
### 13. Additional Relevant Skills ($A$)

Rewards valuable domain competencies beyond the explicit requirements without score inflation:
1. Extra candidate skills: $\text{Extra} = \text{Resume Skills} \setminus \text{JD Requirements}$
2. Filtered by activated role domains via `get_relevant_skills_for_jd(jd_skills)`.
3. Normalized against a maximum consideration cap ($M = 5$):
$$A = \min\left(100.0, \frac{|\text{Additional Relevant}|}{5} \times 100.0\right)$$


In [ ]:
print(f"Additional Relevant Skills ({len(skill_match_res['additional_skills'])}): {skill_match_res['additional_skills']}")
print(f"Additional Score (A): {skill_match_res['additional_score']:.2f}%")


---
### 14. Final Composite Scoring Formula

SkillSync applies the verified formula:
$$\text{Final Score} = 0.70 \times R + 0.10 \times A + 0.20 \times T$$

| Component | Weight | Meaning |
| :--- | :---: | :--- |
| **$R$ (Required Match)** | **70%** | Percentage of mandatory JD requirements present in resume |
| **$A$ (Additional Relevant)** | **10%** | Relevant role proficiencies beyond mandatory requirements |
| **$T$ (TF-IDF Similarity)** | **20%** | Vocabulary and contextual alignment between texts |


In [ ]:
R = skill_match_res['skill_score']
A = skill_match_res['additional_score']
T = sim_score
final_calc = 0.70 * R + 0.10 * A + 0.20 * T
verdict = get_verdict(final_calc)

print(f"Formula Calculation:")
print(f"0.70 * {R:.2f}% + 0.10 * {A:.2f}% + 0.20 * {T:.2f}%")
print(f"= {final_calc:.2f}% [{verdict}]")


---
### 15. Benchmark Demonstration: Full Analysis Across Multiple Roles

We evaluate the candidate resume across 3 different technical roles and 1 negative control role.


In [ ]:
benchmark_jds = {
    '1. Full Stack Developer': SAMPLE_JD,
    '2. Python Backend Engineer': '''
We need a Python Backend Developer proficient in Python, FastAPI, Docker, PostgreSQL, Redis, AWS, and REST APIs.
''',
    '3. Systems & HPC Engineer': '''
HPC Engineer: C, C++, OpenMP, BFS, Parallel Computing, Synchronization, Cache Coherence, Benchmarking.
''',
    '4. Clinical Healthcare (Negative Control)': '''
Registered Nurse / Clinical Specialist: Patient care, CPR, BLS, triage, medical records, vital signs.
'''
}

results_table = []
for role_name, jd_text in benchmark_jds.items():
    res = calculate_final_score(SAMPLE_RESUME, jd_text)
    results_table.append({
        'Role': role_name,
        'Final Score': f"{res['final_score']:.2f}%",
        'Verdict': get_verdict(res['final_score']),
        'R (Req)': f"{res['skill_score']:.2f}%",
        'A (Addl)': f"{res['additional_score']:.2f}%",
        'T (TF-IDF)': f"{res['tfidf_score']:.2f}%",
        'Matched Count': len(res['matched_skills']),
        'Missing Count': len(res['missing_skills']),
    })

df_results = pd.DataFrame(results_table)
df_results


---
### 16. Model Serialization (`models/skillsync_model.pkl`)

**Academic Context:** SkillSync does not employ a black-box supervised classification model. Instead, its core NLP components comprise:  
1. A fitted domain `TfidfVectorizer` capturing n-gram feature representations from professional resumes.  
2. The deterministic canonical taxonomy and alias lookup mappings.  
3. Role relevance clusters (`ROLE_TAXONOMY`).  

We package and serialize these components into `models/skillsync_model.pkl` along with explicit version and environment metadata.


In [ ]:
model_dir = Path('models')
model_dir.mkdir(exist_ok=True)
pickle_file = model_dir / 'skillsync_model.pkl'

# Load representative domain corpus
csv_path = Path('data/resume-data/Resume/Resume.csv')
if csv_path.exists():
    df_resumes = pd.read_csv(csv_path, usecols=['Resume_str'])
    corpus_sample = df_resumes['Resume_str'].dropna().sample(200, random_state=42).tolist()
    cleaned_corpus = [clean_text(t) for t in corpus_sample]
else:
    cleaned_corpus = [clean_text(SAMPLE_RESUME), clean_text(SAMPLE_JD)]

# Fit domain TF-IDF model
domain_vectorizer = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True, max_features=5000)
domain_vectorizer.fit(cleaned_corpus)

# Assemble artifact payload
model_artifact = {
    'metadata': {
        'model_name': 'SkillSync NLP Resume Intelligence Model',
        'version': '2.1.0',
        'python_version': sys.version.split()[0],
        'sklearn_version': sklearn.__version__,
        'formula': '0.70*R + 0.10*A + 0.20*T',
        'description': 'Fitted TF-IDF domain vectorizer, canonical taxonomy, and role relevance clusters.'
    },
    'vectorizer': domain_vectorizer,
    'vocabulary_size': len(domain_vectorizer.vocabulary_),
    'taxonomy': {
        'canonical_skills': SKILLS,
        'skill_aliases': SKILL_ALIASES,
        'role_taxonomy': ROLE_TAXONOMY,
    }
}

with open(pickle_file, 'wb') as f:
    pickle.dump(model_artifact, f)

size_kb = pickle_file.stat().st_size / 1024
print(f"Successfully serialized artifact to: {pickle_file}")
print(f"Artifact Size: {size_kb:.2f} KB")
print(f"Fitted Vocabulary Size: {len(domain_vectorizer.vocabulary_)} terms")


---
### 17. Model Verification & Artifact Deserialization

We reload `models/skillsync_model.pkl` in a clean Python context to verify object integrity, feature shape, and metadata consistency.


In [ ]:
with open('models/skillsync_model.pkl', 'rb') as f:
    loaded_model = pickle.load(f)

meta = loaded_model['metadata']
loaded_vec = loaded_model['vectorizer']
loaded_tax = loaded_model['taxonomy']

print("=== LOADED ARTIFACT METADATA ===")
for k, v in meta.items():
    print(f"  {k:18}: {v}")

print("\n=== VOCABULARY VALIDATION ===")
print(f"Vocabulary size: {len(loaded_vec.vocabulary_)} terms")

# Perform test inference/transformation
test_doc = ["python developer with fastapi and docker experience"]
doc_vec = loaded_vec.transform(test_doc)
print(f"Transformed vector shape: {doc_vec.shape} (Sparse matrix with {doc_vec.nnz} non-zero entries)")

# Validate taxonomy contents
print(f"Taxonomy canonical skills count: {len(loaded_tax['canonical_skills'])}")
print(f"Taxonomy aliases count:          {len(loaded_tax['skill_aliases'])}")
print(f"Taxonomy role domains:           {list(loaded_tax['role_taxonomy'].keys())}")

assert len(loaded_vec.vocabulary_) > 0, "Vocabulary must not be empty!"
assert 'web_fullstack' in loaded_tax['role_taxonomy'], "Taxonomy role missing!"
print("\n[PASSED] All pickle verification assertions succeeded!")


---
### 18. Results & Conclusion

#### Summary of Findings:
1. **Deterministic Accuracy:** SkillSync successfully eliminates misleading 100% matches by evaluating multiword phrases, boundary conditions, and exact skill gaps.
2. **Role Relevance Without Inflation:** Component $A$ (10%) rewards valuable adjacent competencies while ensuring completely unrelated skills receive zero additional credit.
3. **Transparent Serialization:** The project model artifact (`models/skillsync_model.pkl`) serializes fitted n-gram TF-IDF representations alongside the full skill taxonomy, ensuring reproducible offline execution.
4. **Production Architecture:** The underlying modules cleanly integrate with the Streamlit frontend, automated test suites (84 passing pytest tests), and Docker deployment containers.

**End of Notebook.**
